# GS iteration comparison (agent diagnostic)

Compare **pyPhasemasks** vs **pyGS** GS update path with **filter OFF**, same `downscale`, fixed iterations (no early-stop analysis).

Goal: find why masks differ at the same iteration index when scatter/`E`/`A` updates are supposed to match.

**Note:** This notebook does not modify `pyGS.py` or `pyPhasemasks.py`. See findings below.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import cupy as cp
import numpy as np
import matplotlib.pyplot as plt

_root = Path.cwd()
_algorithms = _root.parent.parent if _root.name == "test" else _root.parent
_gs_dir = _algorithms / "GS"
for p in (_algorithms, _gs_dir):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from GS.legacy import pyPhasemasks
from pyGS import GS
from toolbox.gs_operators import complex_field, power_field, normalize_field, create_field
from toolbox.setup import optical_setup, GS_mask_setup

In [ ]:
# Shared params — align both engines (filter OFF)
DOWNscale = 2
MASK_IDX = 0
MAX_STEPS = 80

SHARED = dict(
    wavelength=1.3,
    fin=25e3,
    fout=8e3,
    masksize=960,
    slm_pixel_size=9.2,
    MMF_mfd=12.2,
    downscale=DOWNscale,
    upsample=0,
    MAX_iterations=300,
)

data = np.load(_root / "data_testing/modal_decompositions/faces.npz")
sad = data["sad"][0].astype(np.complex64)
Target = np.empty((1, sad.shape[0]), np.complex64)
Target[0, :] = sad

In [ ]:
def rel_err(a, b):
    a = cp.asarray(a)
    b = cp.asarray(b)
    num = float(cp.sqrt(cp.sum(cp.abs(a - b) ** 2)))
    den = float(cp.sqrt(cp.sum(cp.abs(b) ** 2)))
    return num / max(den, 1e-30)


print("Building OLD env (pyPhasemasks)...")
old = pyPhasemasks.SetupGenericMaskGSenvioment(
    wavelength=SHARED["wavelength"], fin=SHARED["fin"], fout=SHARED["fout"],
    masksize=SHARED["masksize"], slm_pixel_size=SHARED["slm_pixel_size"],
    MMF_mfd=SHARED["MMF_mfd"], MMFdiameter=62.5, MaxModegroups=22,
    maskscount=1, goal_fidelity=0.99, downscale=SHARED["downscale"],
    upsample=SHARED["upsample"], MAX_iterations=SHARED["MAX_iterations"],
)

print("Building NEW env (pyGS)...")
optical = optical_setup(
    wavelength=SHARED["wavelength"], fin=SHARED["fin"], fout=SHARED["fout"],
    masksize=SHARED["masksize"], slm_pixel_size=SHARED["slm_pixel_size"],
    MMF_mfd=SHARED["MMF_mfd"], MMF_diameter=62.5, MMF_mode_groups=22, mode_basis="LG",
)
masksetup = GS_mask_setup(
    num_masks=1, downscale=SHARED["downscale"], upsample=SHARED["upsample"],
    MAX_iterations=SHARED["MAX_iterations"], filtering=False,
)
new = GS(optical, masksetup, engine_mode="fields", number_masks=1)

print(f"Grid: old SLM_PLANE_N={old.SLM_PLANE_N}, new={new.slmplane.SLM_PLANE_N}")

## 1. Static catalog check (before GS loop)

In [ ]:
B_old = old.ILLUMINATION_gpu
B_new = new.gpu_memory.get("ILLUMINATION")
R0_old = old.R0_gpu
R0_new = new.gpu_memory.get("R0")
M_old = old.Modes1D_bases
M_new = new.gpu_memory.get("MODES_ARRAY_1D")

catalog = {
    "ILLUMINATION": rel_err(B_old, B_new),
    "R0": rel_err(R0_old, R0_new),
    "MODES_ARRAY_1D": rel_err(M_old, M_new),
}
for k, v in catalog.items():
    print(f"{k:20s} rel_L2 = {v:.3e}")

B_old_np = cp.asnumpy(B_old)
B_new_np = cp.asnumpy(B_new)
print(f"\nOLD sum|B|^2 = {np.sum(np.abs(B_old_np)**2):.4f}")
print(f"NEW sum|B|^2 = {np.sum(np.abs(B_new_np)**2):.4f}")
print(f"OLD peak at {np.unravel_index(np.argmax(np.abs(B_old_np)), B_old_np.shape)}")
print(f"NEW peak at {np.unravel_index(np.argmax(np.abs(B_new_np)), B_new_np.shape)}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].imshow(np.abs(B_old_np), cmap="viridis")
ax[0].set_title("OLD |ILLUMINATION|")
ax[1].imshow(np.abs(B_new_np), cmap="viridis")
ax[1].set_title("NEW |ILLUMINATION|")
ax[2].imshow(np.abs(B_old_np - B_new_np), cmap="magma")
ax[2].set_title("|difference|")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

## 2. Illumination normalization bug check (`toolbox/setup.py`)

`illumination.define_IlluminationBeam` uses bare `sum(...)` but `setup.py` only imports `arange, meshgrid, sqrt, zeros` from numpy — **not** `sum`.

Python `sum()` on a 2D array sums **rows**, so normalization is wrong (total power ~ `N` instead of `1`).

In [ ]:
from toolbox.setup import illumination as IllumClass
from numpy import sqrt

R = new.slmplane.R
w0 = 1.9450254011992316e3
slm_ill = np.exp(-(R / w0) ** 2)

wrong_norm = slm_ill / sqrt(sum(np.abs(slm_ill) ** 2))          # what setup.py does now
right_norm = slm_ill / np.sqrt(np.sum(np.abs(slm_ill) ** 2))    # what pyPhasemasks does (pylab sum)

print("wrong total power:", np.sum(np.abs(wrong_norm) ** 2))
print("right total power:", np.sum(np.abs(right_norm) ** 2))
print("matches NEW engine power:", np.isclose(np.sum(np.abs(B_new_np) ** 2), np.sum(np.abs(wrong_norm) ** 2), rtol=0.01))

## 3. Iteration-by-iteration GS update (filter OFF, manual loop)

Skips convergence `break` — runs `MAX_STEPS` updates. Compares `A`, `D`, cropped mask each step.

In [ ]:
coefs = cp.asarray(Target[MASK_IDX])
N = old.SLM_PLANE_N

# --- init (both paths) ---
tf_old = cp.matmul(coefs, M_old)
tf_old = tf_old / cp.sqrt(cp.sum(cp.abs(tf_old) ** 2))
Target_Mode = cp.reshape(tf_old, (N, N))
A_old = cp.fft.fftshift(cp.fft.ifft2(cp.fft.fftshift(Target_Mode)))
A_old = A_old / cp.sqrt(cp.sum(cp.abs(A_old) ** 2))

tf_new = cp.matmul(coefs, M_new)
target_field = normalize_field(cp.reshape(tf_new, (N, N)), power_field(cp.reshape(tf_new, (N, N))))
A_new = normalize_field(
    cp.fft.fftshift(cp.fft.ifft2(cp.fft.fftshift(target_field))),
    power_field(cp.fft.fftshift(cp.fft.ifft2(cp.fft.fftshift(target_field)))),
)

print("A_init rel_L2:", rel_err(A_old, A_new))

R1_old = cp.negative(R0_old) + 1
R1_new = cp.negative(R0_new) + 1
crop = old.SLM_N // 2
c = old.SLM_center
MH_old = old.Modes1D_bases_conj
MH_new = new.gpu_memory.get("MODES_ARRAY_1D_HERMITIAN")

w_old = w_new = 1.0
rows = []

for i in range(MAX_STEPS):
    C_old = cp.multiply(cp.abs(B_old), cp.exp(1j * cp.angle(A_old)))
    D_old = cp.fft.fftshift(cp.fft.fft2(cp.fft.fftshift(C_old)))
    D_old = D_old / cp.sqrt(cp.sum(cp.abs(D_old) ** 2))

    C_new = complex_field(B_new, A_new)
    D_new = cp.fft.fftshift(cp.fft.fft2(cp.fft.fftshift(C_new)))
    D_new = normalize_field(D_new, power_field(D_new))

    D1d_old = cp.reshape(D_old, -1).astype(cp.complex64)
    D1d_new = cp.reshape(D_new, -1).astype(cp.complex64)
    eff_old = float(cp.sum(cp.abs(cp.matmul(D1d_old, MH_old)) ** 2))
    eff_new = float(power_field(cp.matmul(D1d_new, MH_new)))

    mask_old = C_old[c - crop : c + crop, c - crop : c + crop]
    mask_new = C_new[c - crop : c + crop, c - crop : c + crop]

    rows.append({
        "i": i,
        "rel_A": rel_err(A_old, A_new) if i > 0 else 0.0,
        "rel_D": rel_err(D_old, D_new),
        "rel_mask": rel_err(mask_old, mask_new),
        "eff_old": eff_old,
        "eff_new": eff_new,
    })

    # OLD update, filter OFF
    scatter_old = (D_old * R1_old) / cp.sqrt(cp.sum(cp.abs(D_old * R1_old) ** 2))
    E_old = (Target_Mode * R0_old / cp.sqrt(cp.sum(cp.abs(Target_Mode * R0_old) ** 2))) * w_old + cp.sqrt(1 - w_old) * scatter_old
    A_old = cp.fft.fftshift(cp.fft.ifft2(cp.fft.fftshift(E_old)))
    w_old *= 0.99

    # NEW update, filter OFF
    scatter_new = normalize_field(D_new * R1_new, power_field(D_new * R1_new))
    E_core = target_field * R0_new
    E_new = create_field(E_core, cp.sqrt(power_field(E_core)), cp.sqrt(1 - w_new) * scatter_new, w_new)
    A_new = cp.fft.fftshift(cp.fft.ifft2(cp.fft.fftshift(E_new)))
    w_new *= 0.99

import pandas as pd
df = pd.DataFrame(rows)
display(df.head(10))
display(df.tail(5))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(df["i"], df["rel_D"], label="rel_D")
ax[0].plot(df["i"], df["rel_mask"], label="rel_mask")
ax[0].set_xlabel("iteration")
ax[0].set_ylabel("relative L2 error")
ax[0].legend()
ax[0].set_title("Field/mask mismatch vs iteration")

ax[1].plot(df["i"], df["eff_old"], label="old efficiency")
ax[1].plot(df["i"], df["eff_new"], label="new efficiency")
ax[1].set_xlabel("iteration")
ax[1].legend()
ax[1].set_title("Efficiency vs iteration")
plt.tight_layout()
plt.show()

## 4. Findings (what was off)

### Root cause: **ILLUMINATION beam in `toolbox/setup.py` is wrong**

In `illumination.define_IlluminationBeam` (`toolbox/setup.py` ~line 184):

```python
slm_ill = slm_ill / sqrt(sum(abs(slm_ill)**2))
```

`setup.py` imports `sqrt` from numpy but **not** `sum`. Bare `sum()` is Python's row-sum on 2D arrays.

| | pyPhasemasks (old) | pyGS via setup.py (new) |
|--|-------------------|------------------------|
| Normalization | `numpy.sum` → unit power | `builtin sum` → **~N pixels power** |
| `sum|B|^2` | ≈ 1 | ≈ **480** (for 480×480 grid) |
| Peak location | center | **off-center** (wrong broadcast) |
| `rel_L2` vs old | 0 | **~0.97** at iteration 0 |

Because `C = |B| exp(i arg A)` uses **B = ILLUMINATION**, `D` and the mask already differ at **iteration 0** even when:
- `R0` matches (rel ≈ 0)
- `MODES` match (rel ≈ 0)
- `A_init` matches (rel ≈ 0)
- filter is OFF
- scatter/`E`/`create_field` algebra matches old code

### Not the cause (for filter-off iteration mismatch)

- `gs_operators` (`complex_field`, `normalize_field`, `create_field`) — equivalent to old inline math
- `R1 = 1 - R0` scatter masking — correct in both
- Convergence metric / early stop — irrelevant to per-step update path (this notebook uses manual loop)
- Gaussian filter order — only matters when `filter=True`

### Fix (when you choose to apply it)

In `toolbox/setup.py` `define_IlluminationBeam`, change to:

```python
slm_ill = slm_ill / np.sqrt(np.sum(np.abs(slm_ill) ** 2))
```

Or add `sum` to the numpy imports at top of `setup.py` (same as pyPhasemasks `from pylab import *`).

### Helper script

Same analysis without the notebook: `test/gs_iteration_compare.py`